# 05 - Empirical Analysis & Primary Thesis Publication Figures

**Active Publication Figures Pipeline**

This notebook executes the **primary empirical analysis** and renders the **active publication figures** for the thesis:
1. **Section 1: Environment & Trace Ingestion** — Initializes statistical services, paths, ingests SQLite synthesis records and IOH evaluation traces, and prepares the global AUC-ECDF performance matrix.
2. **Section 2: Core Thesis Publication Figures**:
   - **Figure 1 (RQ1)**: Benchmark Difficulty Shift under Stochastic Noise (`fig_01`).
   - **Figure 9D (RQ1)**: Solver × BBOB Problem Function Performance Matrix (`fig_09d`).
   - **Figure 9E (Sub-RQ A)**: LLM Parameter Scale Ablation (7B vs. 14B) Across Dimensions (`fig_09e`).
   - **Model Hardness Breakdown**: Separable vs. Conditioning vs. Multi-Modal success rates.
3. **Section 3: Search Dynamics & Empirical Runtime ECDFs**:
   - Multi-panel log-error convergence curves with IQR ribbons and runtime ECDFs.
   - Single-algorithm cross-environment direct noise overlay plots.
4. **Section 4: Algorithmic Failure Analysis**:
   - **Figure 10**: Multi-Tier Convergence Breakdown into Stagnation, Divergence, and High-Precision Tiers.

*(Note: Inferential non-parametric hypothesis testing, noise robustness & landscape fragility suites (Figures 5, 6, 9C), and exploratory ablations (Figures 3, 4, 7, 9B) have been moved to [`05_legacy_figures.ipynb`](05_legacy_figures.ipynb)).*

---
## Section 1: Environment Setup & Data Ingestion

### 1.1 Environment Setup, Paths & Plotly Theme Initialization
Initializes reporting directories (`results/main_results`, `results/profiles`, `results/failure_analysis`, etc.), service dependencies, and typography standards.

In [9]:
%load_ext autoreload
%autoreload 2

# Ensure project root src/ is in sys.path
import os
import sys
from pathlib import Path

cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from collections import defaultdict
import colorsys
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from shared.config import RESULTS_DIR
from shared.database.engine import create_db_session_factory
from benchmarking.infra.storage import SQLiteSynthesisReadRepository
from benchmarking.infra.io.trace_repository import IOHTraceReader
from benchmarking.application.statistical_service import StatisticalEvaluationService
from benchmarking.domain.enums import BBOBFunction
from benchmarking.domain import EvaluationCondition, EvaluationDataset, RunTrace
from benchmarking.domain.services.resolvers import (
    resolve_canonical_model_slug,
    resolve_folder_solver_name,
)
# ─── Thesis Visualization Palette & Dynamic Styling Engine ────────────────────
FONT_FAMILY = "Inter, -apple-system, BlinkMacSystemFont, Arial, sans-serif"

STRATEGY_COLOR_ARCHETYPES = {
    "guided": {"large": "#38BDF8", "small": "#38BDF8", "base": "#38BDF8"},
    "thinking": {"large": "#34D399", "small": "#34D399", "base": "#34D399"},
    "vectorization": {"large": "#F87171", "small": "#F87171", "base": "#F87171"},
    "baseline": {"large": "#FBBF24", "small": "#FBBF24", "base": "#FBBF24"},
}
STRATEGY_PALETTE = {k: v["base"] for k, v in STRATEGY_COLOR_ARCHETYPES.items()}

CLASSICAL_SOLVERS_STYLE = {
    "cma-es": {"color": "#334155", "dash": "dash", "width": 2.2, "name": "CMA-ES"},
    "pso": {"color": "#0D9488", "dash": "dashdot", "width": 2.2, "name": "PSO"},
    "de": {"color": "#7C3AED", "dash": "dot", "width": 2.2, "name": "DE"},
}

REGIME_PALETTE = {
    "clean": {"color": "#38BDF8", "border": "rgba(15, 23, 42, 0.4)", "name": "Clean (σ=0.0)", "pattern": None},
    "noisy": {"color": "#FB923C", "border": "rgba(124, 45, 18, 0.4)", "name": "Noisy (σ=0.05)", "pattern": {"shape": "/", "fillmode": "replace", "fgcolor": "#FFFFFF", "fgopacity": 0.35, "size": 6}},
}

DIMENSION_PALETTE_CLEAN = {2: "#BAE6FD", 3: "#7DD3FC", 5: "#38BDF8", 10: "#0284C7"}
DIMENSION_PALETTE_NOISY = {2: "#FED7AA", 3: "#FDBA74", 5: "#FB923C", 10: "#F97316"}

MODEL_SCALE_PALETTE = {
    "Qwen2.5-Coder-3B": "#BAE6FD",
    "Qwen2.5-Coder-7B": "#7DD3FC",
    "Qwen2.5-Coder-14B": "#0284C7",
    "Qwen2.5-Coder-32B": "#1E3A8A",
}


def hsl_to_hex(h: float, s: float, lightness: float) -> str:
    r, g, b = colorsys.hls_to_rgb(h, lightness, s)
    return "#{:02X}{:02X}{:02X}".format(
        int(round(max(0.0, min(1.0, r)) * 255)),
        int(round(max(0.0, min(1.0, b)) * 255)),
        int(round(max(0.0, min(1.0, b)) * 255)),
    )


def get_model_scale_color(model_name: str) -> str:
    if model_name in MODEL_SCALE_PALETTE:
        return MODEL_SCALE_PALETTE[model_name]
    hue = (abs(hash(model_name)) * 0.618033988749895) % 1.0
    return hsl_to_hex(hue, s=0.80, lightness=0.45)


_DYNAMIC_STYLE_CACHE = {}


def get_solver_line_style(solver_name: str) -> dict:
    if not solver_name:
        return {"color": "#64748B", "dash": "solid", "width": 2.0}
    s = solver_name.strip()
    s_lower = s.lower()
    if s in _DYNAMIC_STYLE_CACHE:
        return _DYNAMIC_STYLE_CACHE[s]
    if s_lower in CLASSICAL_SOLVERS_STYLE:
        res = CLASSICAL_SOLVERS_STYLE[s_lower].copy()
        _DYNAMIC_STYLE_CACHE[s] = res
        return res
    if " / " in s:
        model_part, strat_part = s.split(" / ", 1)
        is_adapted = "(noise-adapted)" in strat_part.lower()
        clean_strat = strat_part.lower().replace("(noise-adapted)", "").strip()
        size_match = re.search(r"(\d+(?:\.\d+)?)\s*[bB]", model_part)
        is_large = float(size_match.group(1)) >= 14.0 if size_match else True
        scale_key = "large" if is_large else "small"
        line_width = 2.5 if is_large else 1.8
        if clean_strat in STRATEGY_COLOR_ARCHETYPES:
            hex_color = STRATEGY_COLOR_ARCHETYPES[clean_strat][scale_key]
        else:
            base_hue = (abs(hash(clean_strat)) * 0.618033988749895) % 1.0
            hex_color = hsl_to_hex(base_hue, s=0.85, lightness=0.45 if is_large else 0.62)
        dash_style = "dash" if is_adapted else "solid"
        res = {"color": hex_color, "dash": dash_style, "width": line_width}
        _DYNAMIC_STYLE_CACHE[s] = res
        return res
    hue = (abs(hash(s_lower)) * 0.618033988749895) % 1.0
    hex_color = hsl_to_hex(hue, s=0.75, lightness=0.50)
    res = {"color": hex_color, "dash": "dash", "width": 2.0}
    _DYNAMIC_STYLE_CACHE[s] = res
    return res


def get_solver_color(solver_name: str) -> str:
    return get_solver_line_style(solver_name)["color"]


def get_rgba_fill(hex_color: str, opacity: float = 0.12) -> str:
    if hex_color.startswith("#") and len(hex_color) == 7:
        r = int(hex_color[1:3], 16)
        g = int(hex_color[3:5], 16)
        b = int(hex_color[5:7], 16)
        return f"rgba({r}, {g}, {b}, {opacity})"
    return f"rgba(100, 116, 139, {opacity})"


def get_dimension_color(dim: int, is_noisy: bool = False) -> str:
    if is_noisy:
        return DIMENSION_PALETTE_NOISY.get(dim, "#FB923C")
    return DIMENSION_PALETTE_CLEAN.get(dim, "#3B82F6")


def build_dynamic_solver_palette(solvers) -> dict:
    return {s: get_solver_color(s) for s in solvers}


class DynamicSolverPalette(dict):
    def __getitem__(self, key: str) -> str:
        return get_solver_color(key)

    def get(self, key, default=None):
        if not key:
            return default or "#64748B"
        return get_solver_color(str(key))


SOLVER_PALETTE = DynamicSolverPalette()
SOLVER_LINE_STYLES = {}

session_factory = create_db_session_factory()
sqlite_repo = SQLiteSynthesisReadRepository(session_factory)
trace_reader = IOHTraceReader()
service = StatisticalEvaluationService(sqlite_repo=sqlite_repo, trace_repo=trace_reader)

# ── 1. Structured Results Subdirectories ─────────────────────────────────
MAIN_RESULTS_DIR  = RESULTS_DIR / "main_results"
FAILURE_DIR       = RESULTS_DIR / "failure_analysis"
CROSS_EVAL_DIR    = RESULTS_DIR / "cross_evaluation"
PROFILES_DIR      = RESULTS_DIR / "profiles"

for d in [MAIN_RESULTS_DIR, FAILURE_DIR, CROSS_EVAL_DIR, PROFILES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REPORTS_DIR       = RESULTS_DIR / "reports"
EVALUATIONS_DIR   = RESULTS_DIR / "ioh_traces"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

FILTER_DIMS = None
FILTER_PROBLEMS = None

def get_noise_color(noise_std: float, all_stds: list[float] | None = None) -> str:
    """Returns an eye-friendly soft color for any given noise level."""
    if noise_std == 0.0:
        return "#38BDF8"  # Soft sky blue for clean
    if all_stds is None or len(all_stds) <= 2:
        if np.isclose(noise_std, 0.05):
            return "#FB923C"  # Soft amber / peach
        elif np.isclose(noise_std, 0.1):
            return "#F87171"  # Soft coral
        elif np.isclose(noise_std, 0.2):
            return "#C084FC"  # Soft lavender
        return "#FB923C"
    
    noisy_stds = sorted([s for s in all_stds if s > 0.0])
    if not noisy_stds:
        return "#FB923C"
    idx = noisy_stds.index(noise_std) if noise_std in noisy_stds else 0
    palette = ["#FDBA74", "#FB923C", "#F87171", "#FB7185", "#E879F9", "#C084FC", "#A78BFA"]
    return palette[idx % len(palette)]

FILTER_NOISE_STDS = None

print("✅ Statistical service and unified dynamic thesis visualization palette initialized.")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Statistical service and unified dynamic thesis visualization palette initialized.


### 1.2 Ingest Benchmark Traces & Synthesis Database Records
Loads all completed evolutionary synthesis experiments from SQLite (`data/db.sqlite3`) and all empirical evaluation traces ($N=20$) from `results/ioh_traces/`.

In [10]:
# Ingest benchmark traces and synthesis records
df_exp, df_iter = service.get_synthesis_dataframes()
all_benchmark_data = service.load_evaluation_traces(
    dims=FILTER_DIMS,
    problems=FILTER_PROBLEMS,
    noise_stds=FILTER_NOISE_STDS,
    solver_resolver=resolve_folder_solver_name,
)

if not all_benchmark_data:
    raise RuntimeError(f'No benchmark traces found in {EVALUATIONS_DIR}!')

all_dims = all_benchmark_data.dims
all_noise_stds = all_benchmark_data.noise_stds
clean_std = 0.0 if 0.0 in all_noise_stds else (all_noise_stds[0] if all_noise_stds else 0.0)
noisy_std = next((n for n in all_noise_stds if n > 0.0), all_noise_stds[-1] if all_noise_stds else 0.05)
PROBLEM_IDS = all_benchmark_data.problem_ids

DISCOVERED_SOLVERS = all_benchmark_data.solvers
SOLVER_PALETTE = build_dynamic_solver_palette(DISCOVERED_SOLVERS)

MODELS_TO_SOLVERS = defaultdict(list)
for s in DISCOVERED_SOLVERS:
    if ' / ' in s:
        MODELS_TO_SOLVERS[s.split(' / ')[0]].append(s)

LLM_SOLVERS_ORDER = [s for s in DISCOVERED_SOLVERS if ' / ' in s]
CLASSICAL_SOLVERS_ORDER = [s for s in DISCOVERED_SOLVERS if ' / ' not in s]
ALL_SOLVERS_ORDER = LLM_SOLVERS_ORDER + CLASSICAL_SOLVERS_ORDER

print(f'📦 Loaded {len(df_exp)} experiments and {len(all_benchmark_data)} problem conditions.')
print(f'🎯 Problems: {PROBLEM_IDS} | Dimensions: {all_dims} | Solvers: {DISCOVERED_SOLVERS}')


📦 Loaded 3550 experiments and 80 problem conditions.
🎯 Problems: [1, 8, 11, 15, 21] | Dimensions: [2, 3, 5, 10] | Solvers: [<ClassicalSolver.CMA_ES: 'CMA-ES'>, <ClassicalSolver.DE: 'DE'>, <ClassicalSolver.PSO: 'PSO'>, 'Qwen2.5-Coder-14B / baseline', 'Qwen2.5-Coder-14B / baseline (noise-adapted)', 'Qwen2.5-Coder-14B / guided', 'Qwen2.5-Coder-14B / guided (noise-adapted)', 'Qwen2.5-Coder-14B / thinking', 'Qwen2.5-Coder-14B / thinking (noise-adapted)', 'Qwen2.5-Coder-14B / vectorization', 'Qwen2.5-Coder-14B / vectorization (noise-adapted)', 'Qwen2.5-Coder-32B / baseline', 'Qwen2.5-Coder-32B / baseline (noise-adapted)', 'Qwen2.5-Coder-32B / guided', 'Qwen2.5-Coder-32B / guided (noise-adapted)', 'Qwen2.5-Coder-32B / thinking', 'Qwen2.5-Coder-32B / thinking (noise-adapted)', 'Qwen2.5-Coder-32B / vectorization', 'Qwen2.5-Coder-32B / vectorization (noise-adapted)', 'Qwen2.5-Coder-3B / baseline', 'Qwen2.5-Coder-3B / baseline (noise-adapted)', 'Qwen2.5-Coder-3B / guided', 'Qwen2.5-Coder-3B / gui

### 1.3 Compute 51-Target Adaptive Targets & AUC-ECDF Evaluation Matrix
Computes the adaptive precision target grid and evaluates the global AUC-ECDF matrix across all solvers and test conditions required by Figures 9D and 9E.

In [11]:
# ── Compute 51-Target Adaptive Targets & AUC-ECDF Matrix ───────────────────────
targets_dict = {n_std: service.compute_adaptive_targets(all_benchmark_data, noise_std=n_std) for n_std in all_noise_stds}
df_all = service.compute_auc_ecdf_matrix(all_benchmark_data, ALL_SOLVERS_ORDER, targets=targets_dict, group_by="condition")

# Compute canonical solver ranking by grouping across clean/adapted variations
df_all["Canonical Solver"] = df_all["Solver"].str.replace(r" \(noise-adapted\)", "", regex=True)
solver_overall_auc = df_all.groupby("Canonical Solver")["AUC-ECDF (%)"].mean().sort_values(ascending=False)
sorted_solvers_auc = solver_overall_auc.index.tolist()

print(f"✅ Computed AUC-ECDF matrix for {len(df_all)} condition entries across {len(sorted_solvers_auc)} canonical solvers.")


✅ Computed AUC-ECDF matrix for 686 conditions across 19 solvers.


---
## Section 2: Core Thesis Publication Figures

### 2.1 Figure 1 (RQ1): Benchmark Difficulty Shift under Stochastic Noise
**Research Question 1**: *Does the stochastic Gaussian noise extension systematically elevate optimization difficulty across BBOB problem landscapes?*
Generates grouped bar charts comparing final error distributions in clean ($\sigma=0.0$) vs. noisy ($\sigma > 0$) environments per dimension, saved to `results/main_results/fig_01_benchmark_difficulty_{dim}D.png`.

In [12]:
# ── THESIS Figure 1: Benchmark Difficulty under Noise Extension ──────────────
for dim in all_dims:
    benchmark_data_standard = {
        c: {s: runs for s, runs in s_dict.items() if "(noise-adapted)" not in s}
        for c, s_dict in all_benchmark_data.items()
    }
    clean_meds, noisy_meds, p_labels = service.compute_validation_medians(
        benchmark_data_standard, dim, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
    )
    fig1 = go.Figure()
    fig1.add_trace(go.Bar(
        name=f"Deterministic (σ={clean_std})",
        x=p_labels,
        y=np.maximum(clean_meds, 1e-16),
        marker=dict(color=REGIME_PALETTE["clean"]["color"], line=dict(color=REGIME_PALETTE["clean"]["border"], width=1.0))
    ))
    fig1.add_trace(go.Bar(
        name=f"Noisy Stochastic (σ={noisy_std})",
        x=p_labels,
        y=np.maximum(noisy_meds, 1e-16),
        marker=dict(
            color=REGIME_PALETTE["noisy"]["color"],
            pattern=dict(shape="/", fillmode="replace", fgcolor="#FFFFFF", fgopacity=0.4, size=8),
            line=dict(color=REGIME_PALETTE["noisy"]["border"], width=1.0)
        )
    ))
    fig1.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Figure 1: Benchmark Problem Difficulty under Stochastic Noise — {dim}D</b><br><span style=\"font-size:13px;color:#475569;font-weight:normal;\">Median Terminal Optimization Error (Δy) Across All Solvers by BBOB Landscape Class</span>",
            font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
            x=0.02, y=0.96
        ),
        xaxis=dict(
            title="<b>BBOB Landscape Class</b>",
            title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"),
            tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B")
        ),
        yaxis=dict(
            type="log",
            title="<b>Median Final Error log₁₀(Δy)</b>",
            title_font=dict(size=16, family=FONT_FAMILY, color="#0F172A"),
            tickfont=dict(size=14, family=FONT_FAMILY, color="#1E293B"),
            range=[-16, 4],
            showgrid=True, gridwidth=1, gridcolor="#F1F5F9"
        ),
        barmode="group", bargap=0.25, bargroupgap=0.1,
        width=1160, height=620,
        margin=dict(l=85, r=40, t=110, b=110),
        legend=dict(
            orientation="h", yanchor="top", y=-0.16, xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1,
            font=dict(size=14, family=FONT_FAMILY)
        )
    )
    out_p = MAIN_RESULTS_DIR / f"fig_01_benchmark_difficulty_{dim}D.png"
    fig1.write_image(str(out_p), scale=3)

print("✅ Figure 1 (Benchmark Difficulty) generated in results/main_results/")


2026-09-13 10:55:45 INFO TemporaryDirectory.cleanup() worked.
2026-09-13 10:55:45 INFO shutil.rmtree worked.
2026-09-13 10:55:45 INFO Chromium init'ed with kwargs {}
2026-09-13 10:55:45 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-13 10:55:45 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpvn8tlpp4.
2026-09-13 10:55:45 INFO Opening browser.
2026-09-13 10:55:45 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp9ecsrg1r.
2026-09-13 10:55:45 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp9ecsrg1r
2026-09-13 10:55:47 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpvn8tlpp4/index.html
2026-09-13 10:55:48 INFO Getting tab from queue (has 1)
2026-09-13 10:55:48 INFO Got 536D
2026-09-13 10:55:49 INFO Reloading tab 536D before return.
2026-09-13 10:55:49 INFO Putting tab 536D back (queue size: 0).
2026-09-13 10:55:49 

✅ Figure 1 (Benchmark Difficulty) generated in results/main_results/


### 2.2 Figure 9D (RQ1): Solver × BBOB Problem Function Performance Matrix
**Research Question 1**: *Which continuous landscape topologies favor LLM-generated heuristics versus analytical classical baselines?*
Generates a comprehensive AUC-ECDF heatmap across Sphere (f1), Rosenbrock (f8), Discus (f11), Rastrigin (f15), and Gallagher 101 Peaks (f21), saved to `results/main_results/fig_09d_auc_ecdf_by_problem.png`.

In [13]:
# ── THESIS Figure 9D: Solver × Problem Function Matrix (Heatmap) ─────────────
prob_ids = [p for p in [1, 8, 11, 15, 21] if p in PROBLEM_IDS]
prob_labels = [f"{BBOBFunction.get_name(p)} (f{p})" for p in prob_ids]
solvers_y = list(solver_overall_auc.index)

matrix_clean = np.zeros((len(solvers_y), len(prob_ids)))
matrix_noisy = np.zeros((len(solvers_y), len(prob_ids)))
for r_idx, s in enumerate(solvers_y):
    for c_idx, p in enumerate(prob_ids):
        # Clean regime
        c_sub = df_all[(df_all["Canonical Solver"] == s) & (df_all["Problem ID"] == p) & (df_all["Noise Std"] == clean_std)]
        # Noisy regime: matches either clean-transfer or noise-adapted solver
        n_sub = df_all[(df_all["Canonical Solver"] == s) & (df_all["Problem ID"] == p) & (df_all["Noise Std"] == noisy_std)]
        matrix_clean[r_idx, c_idx] = c_sub["AUC-ECDF (%)"].mean() if not c_sub.empty else 0.0
        matrix_noisy[r_idx, c_idx] = n_sub["AUC-ECDF (%)"].mean() if not n_sub.empty else 0.0

fig9d = make_subplots(
    rows=1, cols=2,
    subplot_titles=[f"<b>(A) Clean (σ={clean_std})</b>", f"<b>(B) Noisy (σ={noisy_std})</b>"],
    horizontal_spacing=0.10,
    shared_yaxes=True
)
fig9d.add_trace(go.Heatmap(
    z=matrix_clean, x=prob_labels, y=solvers_y,
    colorscale=[[0.0, "#F8FAFC"], [0.25, "#E0F2FE"], [0.5, "#BAE6FD"], [0.75, "#7DD3FC"], [1.0, "#38BDF8"]], zmin=0, zmax=70,
    text=[[f"{v:.1f}%" if v > 0 else "" for v in row] for row in matrix_clean],
    texttemplate="%{text}",
    textfont=dict(size=12, family=FONT_FAMILY),
    showscale=False,
), row=1, col=1)
fig9d.add_trace(go.Heatmap(
    z=matrix_noisy, x=prob_labels, y=solvers_y,
    colorscale=[[0.0, "#F8FAFC"], [0.25, "#E0F2FE"], [0.5, "#BAE6FD"], [0.75, "#7DD3FC"], [1.0, "#38BDF8"]], zmin=0, zmax=70,
    text=[[f"{v:.1f}%" if v > 0 else "" for v in row] for row in matrix_noisy],
    texttemplate="%{text}",
    textfont=dict(size=12, family=FONT_FAMILY),
    colorbar=dict(
        title="<b>AUC-ECDF (%)</b>",
        title_font=dict(size=14, family=FONT_FAMILY),
        title_side="top",
        tickfont=dict(size=12, family=FONT_FAMILY),
        len=0.85
    ),
), row=1, col=2)

for anno in fig9d.layout.annotations:
    anno.update(font=dict(size=16, color="#0F172A", family=FONT_FAMILY))

plot_h = max(700, len(solvers_y) * 32 + 180)
fig9d.update_layout(
    template="plotly_white",
    title=dict(
        text="<b>Figure 9D: Solver Performance Matrix Across BBOB Problem Landscapes</b><br><span style="font-size:13px;color:#475569;font-weight:normal;">Area Under Runtime ECDF (AUC-ECDF %) Across Canonical Function Classes in Clean vs. Noisy Regimes</span>",
        font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
        x=0.02, y=0.97
    ),
    width=1420, height=plot_h,
    margin=dict(l=220, r=40, t=110, b=90),
)
fig9d.update_xaxes(tickangle=-25, tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"), row=1, col=1)
fig9d.update_xaxes(tickangle=-25, tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"), row=1, col=2)
fig9d.update_yaxes(tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"), autorange="reversed", row=1, col=1)

out_9d = MAIN_RESULTS_DIR / "fig_09d_auc_ecdf_by_problem.png"
fig9d.write_image(str(out_9d), scale=3)
print("✅ Figure 9D (By Problem Heatmap) generated in results/main_results/")


2026-09-13 10:55:54 INFO TemporaryDirectory.cleanup() worked.
2026-09-13 10:55:54 INFO shutil.rmtree worked.
2026-09-13 10:55:54 INFO TemporaryDirectory.cleanup() worked.
2026-09-13 10:55:54 INFO shutil.rmtree worked.
2026-09-13 10:55:54 INFO TemporaryDirectory.cleanup() worked.
2026-09-13 10:55:54 INFO shutil.rmtree worked.
2026-09-13 10:55:54 INFO Chromium init'ed with kwargs {}
2026-09-13 10:55:54 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-13 10:55:54 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp4dhx8_xr.
2026-09-13 10:55:54 INFO Opening browser.
2026-09-13 10:55:54 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpkv6lyots.
2026-09-13 10:55:54 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpkv6lyots
2026-09-13 10:55:54 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp4dhx8_xr/index.html
2026-09-13 10:5

✅ Figure 9D (By Problem Heatmap) generated in results/main_results/


### 2.3 Figure 9E (Sub-RQ A): LLM Parameter Scale Ablation Across Dimensions
Analyzes the empirical performance scaling trajectory across parameter scales (3B, 7B, 14B, 32B) and dimensionality regimes ($D \in \{2, 3, 5, 10\}$) in deterministic ($\sigma = 0.0$) and stochastic noise ($\sigma > 0.0$) conditions relative to classical numerical optimizers.


In [14]:
# ── THESIS Figure 9E: LLM Parameter Scale Ablation Across Dimensions ────────
fig9e = make_subplots(
    rows=1, cols=2,
    subplot_titles=[f"<b>(A) Clean (σ={clean_std})</b>", f"<b>(B) Noisy (σ={noisy_std})</b>"],
    horizontal_spacing=0.10,
    shared_yaxes=True
)
dims = [d for d in [2, 3, 5, 10] if d in all_dims]

# Discover model families dynamically
def extract_model_scale(m: str) -> float:
    match = re.search(r"(\d+(?:\.\d+)?)\s*[bB]", m)
    return float(match.group(1)) if match else 0.0

discovered_models = sorted(list(MODELS_TO_SOLVERS.keys()), key=extract_model_scale)

for col_idx, (n_std, title_sfx) in enumerate([(clean_std, "Clean"), (noisy_std, "Noisy")], start=1):
    for model_name in discovered_models:
        sub_m = df_all[df_all["Solver"].str.startswith(f"{model_name} /")]
        c_m = sub_m[sub_m["Noise Std"] == n_std]
        vals_m = [c_m[c_m["Dim"] == d]["AUC-ECDF (%)"].mean() if not c_m[c_m["Dim"] == d].empty else np.nan for d in dims]
        if any(pd.notna(v) and not np.isnan(v) for v in vals_m):
            m_color = get_model_scale_color(model_name)
            fig9e.add_trace(go.Bar(
                x=[f"{d}D" for d in dims], y=vals_m,
                name=model_name,
                marker=dict(color=m_color, line=dict(color="#0F172A", width=0.8)),
                text=[f"{v:.1f}%" if pd.notna(v) and not np.isnan(v) else "" for v in vals_m], textposition="outside",
                textfont=dict(size=11, family=FONT_FAMILY, color="#1E293B"),
                showlegend=(col_idx == 1)
            ), row=1, col=col_idx)

    for baseline in CLASSICAL_SOLVERS_ORDER:
        b_sub = df_all[(df_all["Solver"] == baseline) & (df_all["Noise Std"] == n_std)]
        b_vals = [b_sub[b_sub["Dim"] == d]["AUC-ECDF (%)"].mean() if not b_sub[b_sub["Dim"] == d].empty else np.nan for d in dims]
        b_style = get_solver_line_style(baseline)
        fig9e.add_trace(go.Scatter(
            x=[f"{d}D" for d in dims], y=b_vals,
            mode="lines+markers", name=baseline,
            line=dict(color=b_style["color"], dash=b_style["dash"], width=2.2),
            marker=dict(size=8, symbol="diamond" if "cma" in baseline.lower() else ("square" if "pso" in baseline.lower() else "circle")),
            showlegend=(col_idx == 1)
        ), row=1, col=col_idx)

for anno in fig9e.layout.annotations:
    anno.update(font=dict(size=16, color="#0F172A", family=FONT_FAMILY))

fig9e.update_layout(
    template="plotly_white",
    title=dict(
        text="<b>Figure 9E: LLM Parameter Scale Ablation Across Dimensions</b><br><span style="font-size:13px;color:#475569;font-weight:normal;">Mean Area Under Runtime ECDF (AUC-ECDF %) Across Synthesized Model Scales vs. Classical Baselines in Clean and Noisy Regimes</span>",
        font=dict(size=20, color="#0F172A", family=FONT_FAMILY),
        x=0.02, y=0.97
    ),
    barmode="group",
    width=1380, height=640,
    margin=dict(l=70, r=40, t=110, b=90),
    legend=dict(
        orientation="h", yanchor="top", y=-0.14, xanchor="center", x=0.5,
        bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1,
        font=dict(size=13, family=FONT_FAMILY)
    )
)
fig9e.update_yaxes(
    title_text="<b>Mean AUC-ECDF (%)</b>",
    title_font=dict(size=15, family=FONT_FAMILY, color="#0F172A"),
    tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"),
    range=[0, 65], showgrid=True, gridcolor="#F1F5F9", row=1, col=1
)
fig9e.update_yaxes(
    tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"),
    range=[0, 65], showgrid=True, gridcolor="#F1F5F9", row=1, col=2
)
fig9e.update_xaxes(
    title_text="<b>Problem Dimension</b>",
    title_font=dict(size=15, family=FONT_FAMILY, color="#0F172A"),
    tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"),
    row=1, col=1
)
fig9e.update_xaxes(
    title_text="<b>Problem Dimension</b>",
    title_font=dict(size=15, family=FONT_FAMILY, color="#0F172A"),
    tickfont=dict(size=13, family=FONT_FAMILY, color="#1E293B"),
    row=1, col=2
)

out_9e = MAIN_RESULTS_DIR / "fig_09e_auc_ecdf_model_scale.png"
fig9e.write_image(str(out_9e), scale=3)
print("✅ Figure 9E (Model Scale Ablation) generated in results/main_results/")


2026-09-13 10:55:55 INFO Chromium init'ed with kwargs {}
2026-09-13 10:55:55 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-13 10:55:55 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpy8lbfhuj.
2026-09-13 10:55:55 INFO Opening browser.
2026-09-13 10:55:55 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp2gi1q3u2.
2026-09-13 10:55:55 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp2gi1q3u2
2026-09-13 10:55:56 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpy8lbfhuj/index.html
2026-09-13 10:55:57 INFO Getting tab from queue (has 1)
2026-09-13 10:55:57 INFO Got 1975
2026-09-13 10:55:57 INFO Reloading tab 1975 before return.
2026-09-13 10:55:57 INFO Putting tab 1975 back (queue size: 0).
2026-09-13 10:55:57 INFO Waiting for all cleanups to finish.
2026-09-13 10:55:57 INFO Exiting Kaleido.
2026-09-13 10:55:57 INFO T

✅ Figure 9E (Model Scale Ablation) generated in results/main_results/


### 2.4 Model-Specific Success Rates by Problem Hardness Class
Separates success rates across Separable, Conditioning, and Multi-Modal function classes for each LLM family in clean vs. noisy regimes, exporting to `results/profiles/`.

In [15]:
# ── Model-Specific Success Rate by Landscape Hardness (Clean vs. Noisy) ──
def render_model_success_rate_by_hardness(model_tag: str, solvers_list: list, dim: int):
    # Check if there is any data for this model in this dimension
    has_model_data = False
    for noise_level in [clean_std, noisy_std]:
        df_hard = service.compute_hardness_success_rates(all_benchmark_data, dim, solvers_list, noise_level=noise_level)
        if not df_hard.empty and any(" / " in s for s in df_hard["Solver"].unique()):
            has_model_data = True
            break
    if not has_model_data:
        return

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f"<b>(A) Deterministic Landscape (σ={clean_std}, {dim}D)</b>",
            f"<b>(B) Noisy Stochastic Landscape (σ={noisy_std}, {dim}D)</b>"
        ),
        horizontal_spacing=0.10
    )
    
    for c_idx, noise_level in enumerate([clean_std, noisy_std], start=1):
        df_hard = service.compute_hardness_success_rates(all_benchmark_data, dim, solvers_list, noise_level=noise_level)
        for solver in solvers_list:
            sub_s = df_hard[df_hard["Solver"] == solver] if not df_hard.empty else pd.DataFrame()
            if not sub_s.empty:
                fig.add_trace(
                    go.Bar(
                        name=solver,
                        x=sub_s["Class"],
                        y=sub_s["Success Rate"],
                        marker=dict(
                            color=get_solver_color(solver),
                            line=dict(color="#0F172A", width=0.8)
                        ),
                        showlegend=(c_idx == 1)
                    ),
                    row=1, col=c_idx
                )
                
    fig.update_xaxes(
        title_text="<b>Landscape Hardness Class</b>",
        title_font=dict(size=14, family=FONT_FAMILY, color="#0F172A"),
        tickangle=-15,
        tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"),
        row=1, col=1
    )
    fig.update_xaxes(
        title_text="<b>Landscape Hardness Class</b>",
        title_font=dict(size=14, family=FONT_FAMILY, color="#0F172A"),
        tickangle=-15,
        tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"),
        row=1, col=2
    )
    fig.update_yaxes(
        title_text="<b>Target Success Rate (Δy ≤ 10⁻⁸)</b>",
        title_font=dict(size=14, family=FONT_FAMILY, color="#0F172A"),
        tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"),
        range=[0, 1.10], showgrid=True, gridcolor="#F1F5F9", row=1, col=1
    )
    fig.update_yaxes(
        tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"),
        range=[0, 1.10], showgrid=True, gridcolor="#F1F5F9", row=1, col=2
    )
    
    for anno in fig.layout.annotations:
        anno.update(font=dict(size=15, color="#0F172A", family=FONT_FAMILY))
        
    fig.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Empirical Success Rate by BBOB Landscape Hardness — {model_tag.upper()} ({dim}D)</b><br><span style="font-size:13px;color:#475569;font-weight:normal;">Comparison of Target Precision Hitting Rates Across 5 Problem Classes in Deterministic vs. Noisy Regimes</span>",
            x=0.02, y=0.96,
            font=dict(size=16, color="#1E293B", family=FONT_FAMILY)
        ),
        barmode="group",
        bargap=0.25,
        bargroupgap=0.08,
        width=1240, height=590,
        margin=dict(l=80, r=40, t=100, b=120),
        legend=dict(
            orientation="h",
            yanchor="top", y=-0.22,
            xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.95)",
            bordercolor="#E2E8F0",
            borderwidth=1,
            font=dict(size=12, family=FONT_FAMILY)
        )
    )
    
    slug = resolve_canonical_model_slug(model_tag)
    m_dir = PROFILES_DIR / slug / f"{dim}D"
    m_dir.mkdir(parents=True, exist_ok=True)
    out_p = m_dir / "figure_success_rate_by_hardness.png"
    fig.write_image(str(out_p), scale=3)

for dim in all_dims:
    for model_name, solvers_list in MODELS_TO_SOLVERS.items():
        solvers_to_plot = solvers_list + CLASSICAL_SOLVERS_ORDER
        render_model_success_rate_by_hardness(model_name, solvers_to_plot, dim)

print("✅ Model-specific success rate by hardness generated for all models and dimensions.")


2026-09-13 10:55:57 INFO Chromium init'ed with kwargs {}
2026-09-13 10:55:57 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-13 10:55:57 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpusdwtrxq.
2026-09-13 10:55:57 INFO Opening browser.
2026-09-13 10:55:57 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpkw05holv.
2026-09-13 10:55:57 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpkw05holv
2026-09-13 10:55:58 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpusdwtrxq/index.html
2026-09-13 10:55:58 INFO TemporaryDirectory.cleanup() worked.
2026-09-13 10:55:58 INFO shutil.rmtree worked.
2026-09-13 10:55:58 INFO TemporaryDirectory.cleanup() worked.
2026-09-13 10:55:58 INFO shutil.rmtree worked.
2026-09-13 10:55:59 INFO Getting tab from queue (has 1)
2026-09-13 10:55:59 INFO Got 122C
2026-09-13 10:55:59 INFO Reloading

✅ Model-specific success rate by hardness generated for all models and dimensions.


---
## Section 3: Search Trajectories & Empirical Runtime ECDFs

### 3.1 Multi-Panel Model Convergence & Empirical Runtime ECDFs
Visualizes optimization trajectory convergence curves (median log-error vs. evaluation budget with IQR ribbons) and runtime ECDFs per problem condition in `results/profiles/`.

In [ ]:
# ── THESIS: Multi-Panel Model Convergence & Empirical Runtime ECDFs ─────────
from benchmarking.infra.storage import EvaluationConfigRepository

config_repo = EvaluationConfigRepository()
bench_cfg = config_repo.load_config()
budget_multiplier = getattr(bench_cfg, "budget_multiplier", 10000)

MAX_EVAL_BUDGET = 1_000_000  # Standardized 10^6 evaluation budget for ECDF and Convergence
eval_grid = np.logspace(0, 6, 300)
tickvals = [1, 10, 100, 1000, 10000, 100000, 1000000]
ticktext = ['1', '10', '100', '1k', '10k', '100k', '1M']

coords = [((i // 3) + 1, (i % 3) + 1) for i in range(len(PROBLEM_IDS) + 1)]
subplot_titles = [f'<b>{BBOBFunction.get_name(p).replace(" Multi-Modal", "")}</b><br><sup>{BBOBFunction.get_class(p)}</sup>' for p in PROBLEM_IDS]
subplot_titles.append('<b>Overall Aggregate Profile</b><br><sup>Mean across 5 BBOB Problem Classes</sup>')

for dim in all_dims:
    for model_name, s_list in MODELS_TO_SOLVERS.items():
        slug = resolve_canonical_model_slug(model_name)
        m_dir = PROFILES_DIR / slug / f'{dim}D'

        for n_std in all_noise_stds:
            # Dynamically select appropriate strategy solvers for this environment
            if n_std == 0.0:
                env_solvers = [s for s in s_list if '(noise-adapted)' not in s]
            else:
                noisy_solvers = [s for s in s_list if '(noise-adapted)' in s]
                clean_transfer = [s for s in s_list if '(noise-adapted)' not in s]
                env_solvers = noisy_solvers if noisy_solvers else clean_transfer

            # Check if this model has any evaluated runs under this specific condition
            has_model_runs = any(
                len(all_benchmark_data.get(EvaluationCondition(dim=dim, noise_std=n_std, problem_id=p_id), {}).get(s_name, [])) > 0
                for p_id in PROBLEM_IDS
                for s_name in env_solvers
            )
            if not has_model_runs:
                continue

            solvers_to_plot = env_solvers + CLASSICAL_SOLVERS_ORDER
            m_dir.mkdir(parents=True, exist_ok=True)
            env_dir = m_dir / f'std_{n_std}'
            env_dir.mkdir(parents=True, exist_ok=True)

            targets = service.compute_adaptive_targets(all_benchmark_data, noise_std=n_std)
            label_env = f"Deterministic (σ=0.0)" if n_std == 0.0 else f"Noisy Stochastic (σ={n_std})"

            # 1. Multi-panel Mean/Median Convergence with IQR Error Bands
            fig_m = make_subplots(
                rows=2, cols=3,
                subplot_titles=subplot_titles,
                vertical_spacing=0.18,
                horizontal_spacing=0.08
            )
            problem_ranges = {}
            for idx, p_id in enumerate(PROBLEM_IDS):
                r_idx, c_idx = coords[idx]
                cond = EvaluationCondition(dim=dim, noise_std=n_std, problem_id=p_id)
                s_dict = all_benchmark_data.get(cond, {})
                p_vals = []
                for s_name in solvers_to_plot:
                    runs = s_dict.get(s_name, [])
                    if runs:
                        mean_t, q25_t, q75_t, _ = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                        s_style = get_solver_line_style(s_name)
                        valid_pts = mean_t[np.isfinite(mean_t) & (mean_t > 0)]
                        if len(valid_pts) > 0:
                            p_vals.extend(valid_pts.tolist())
                        valid_q25 = q25_t[np.isfinite(q25_t) & (q25_t > 0)]
                        if len(valid_q25) > 0:
                            p_vals.extend(valid_q25.tolist())
                        valid_q75 = q75_t[np.isfinite(q75_t) & (q75_t > 0)]
                        if len(valid_q75) > 0:
                            p_vals.extend(valid_q75.tolist())
                        
                        fig_m.add_trace(go.Scatter(
                            x=eval_grid, y=mean_t, mode="lines", name=s_name,
                            line=dict(color=s_style["color"], width=s_style["width"], dash=s_style["dash"]),
                            showlegend=(idx == 0)
                        ), row=r_idx, col=c_idx)
                        
                        hex_c = s_style["color"].lstrip("#")
                        rgb_fill = f"rgba({int(hex_c[0:2],16)}, {int(hex_c[2:4],16)}, {int(hex_c[4:6],16)}, 0.12)" if len(hex_c) == 6 else "rgba(100,100,100,0.12)"
                        fig_m.add_trace(go.Scatter(
                            x=np.concatenate([eval_grid, eval_grid[::-1]]),
                            y=np.concatenate([q75_t, q25_t[::-1]]),
                            fill="toself", fillcolor=rgb_fill,
                            line=dict(color="rgba(255,255,255,0)"),
                            showlegend=False, hoverinfo="skip"
                        ), row=r_idx, col=c_idx)
                if p_vals:
                    p_min = max(float(np.min(p_vals)) * 0.5, 1e-12)
                    p_max = float(np.max(p_vals)) * 2.0
                    p_min_exp = max(np.log10(p_min), -12.0)
                    p_max_exp = min(np.log10(p_max), 8.0)
                    if p_min_exp >= p_max_exp:
                        p_min_exp, p_max_exp = -12.0, 8.0
                else:
                    p_min_exp, p_max_exp = -12.0, 8.0
                problem_ranges[idx] = (p_min_exp, p_max_exp)

            # Panel 6: Overall Aggregate Mean Convergence across all problems
            r_idx6, c_idx6 = coords[len(PROBLEM_IDS)]
            agg_vals = []
            for s_name in solvers_to_plot:
                all_problem_means = []
                for p_id in PROBLEM_IDS:
                    cond = EvaluationCondition(dim=dim, noise_std=n_std, problem_id=p_id)
                    runs = all_benchmark_data.get(cond, {}).get(s_name, [])
                    if runs:
                        mean_t, _, _, _ = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                        all_problem_means.append(mean_t)
                if all_problem_means:
                    mean_med = np.mean(all_problem_means, axis=0)
                    s_style = get_solver_line_style(s_name)
                    fig_m.add_trace(go.Scatter(
                        x=eval_grid, y=mean_med, mode='lines', name=s_name,
                        line=dict(color=s_style['color'], width=s_style['width'], dash=s_style['dash']),
                        showlegend=False
                    ), row=r_idx6, col=c_idx6)
                    valid_pts = mean_med[np.isfinite(mean_med) & (mean_med > 0)]
                    if len(valid_pts) > 0:
                        agg_vals.extend(valid_pts.tolist())

            if agg_vals:
                agg_min = max(float(np.min(agg_vals)) * 0.5, 1e-12)
                agg_max = float(np.max(agg_vals)) * 2.0
                agg_min_exp = max(np.log10(agg_min), -12.0)
                agg_max_exp = min(np.log10(agg_max), 8.0)
                if agg_min_exp >= agg_max_exp:
                    agg_min_exp, agg_max_exp = -12.0, 8.0
            else:
                agg_min_exp, agg_max_exp = -12.0, 8.0
            problem_ranges[len(PROBLEM_IDS)] = (agg_min_exp, agg_max_exp)

            for idx in range(len(PROBLEM_IDS)):
                r, c = coords[idx]
                p_min_exp, p_max_exp = problem_ranges[idx]
                fig_m.update_xaxes(
                    type='log', title_text='<b>Evaluations</b>', title_font=dict(size=13, family=FONT_FAMILY, color="#0F172A"),
                    tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"), range=[0, 6.0],
                    tickvals=tickvals, ticktext=ticktext, showgrid=True, gridcolor='#F1F5F9', row=r, col=c
                )
                fig_m.update_yaxes(
                    type='log', title_text='<b>Mean Error Δy (Log Scale)</b>', title_font=dict(size=13, family=FONT_FAMILY, color="#0F172A"),
                    tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"), range=[p_min_exp, p_max_exp], showgrid=True, gridcolor='#F1F5F9', row=r, col=c
                )

            r6, c6 = coords[len(PROBLEM_IDS)]
            agg_min_exp, agg_max_exp = problem_ranges[len(PROBLEM_IDS)]
            fig_m.update_xaxes(
                type='log', title_text='<b>Evaluations</b>', title_font=dict(size=13, family=FONT_FAMILY, color="#0F172A"),
                tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"), range=[0, 6.0],
                tickvals=tickvals, ticktext=ticktext, showgrid=True, gridcolor='#F1F5F9', row=r6, col=c6
            )
            fig_m.update_yaxes(
                type='log', title_text='<b>Mean Error Δy (Log Scale)</b>', title_font=dict(size=13, family=FONT_FAMILY, color="#0F172A"),
                tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"), range=[agg_min_exp, agg_max_exp], showgrid=True, gridcolor='#F1F5F9', row=r6, col=c6
            )

            for anno in fig_m.layout.annotations:
                anno.update(font=dict(size=15, color='#0F172A', family=FONT_FAMILY))

            fig_m.update_layout(
                template='plotly_white',
                title=dict(
                    text=f'<b>Empirical Convergence Trajectories [{label_env}] — {model_name} ({dim}D)</b><br><span style="font-size:13px;color:#475569;font-weight:normal;">Log-scale Median Convergence with IQR Bands Across 5 BBOB Problem Classes vs. Classical Baselines (Budget = 1,000,000 evals)</span>',
                    font=dict(size=17, color='#0F172A', family=FONT_FAMILY),
                    x=0.02, y=0.97
                ),
                width=1340, height=860,
                margin=dict(l=70, r=40, t=110, b=120),
                legend=dict(
                    orientation='h',
                    yanchor='top', y=-0.14,
                    xanchor='center', x=0.5,
                    bgcolor='rgba(255,255,255,0.95)',
                    bordercolor='#E2E8F0',
                    borderwidth=1,
                    font=dict(size=13, family=FONT_FAMILY)
                )
            )
            out_m_traj = env_dir / 'convergence_trajectories.png'
            fig_m.write_image(str(out_m_traj), scale=3)

            # 2. Multi-panel Empirical Runtime ECDF with Adaptive Targets (5 Problems + Overall Aggregate)
            fig_ecdf = make_subplots(
                rows=2, cols=3,
                subplot_titles=subplot_titles,
                vertical_spacing=0.18,
                horizontal_spacing=0.08
            )
            for idx, p_id in enumerate(PROBLEM_IDS):
                r_idx, c_idx = coords[idx]
                cond = EvaluationCondition(dim=dim, noise_std=n_std, problem_id=p_id)
                s_dict = all_benchmark_data.get(cond, {})
                for s_name in solvers_to_plot:
                    runs = s_dict.get(s_name, [])
                    if runs:
                        _, _, _, ecdf_curve = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                        s_style = get_solver_line_style(s_name)
                        fig_ecdf.add_trace(go.Scatter(
                            x=eval_grid, y=ecdf_curve, mode='lines', name=s_name,
                            line=dict(color=s_style['color'], width=s_style['width'], dash=s_style['dash']),
                            showlegend=(idx == 0)
                        ), row=r_idx, col=c_idx)
                fig_ecdf.update_xaxes(
                    type='log', title_text='<b>Evaluations</b>', title_font=dict(size=13, family=FONT_FAMILY, color="#0F172A"),
                    tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"), range=[0, 6.0],
                    tickvals=tickvals, ticktext=ticktext,
                    showgrid=True, gridcolor='#F1F5F9', row=r_idx, col=c_idx
                )
                fig_ecdf.update_yaxes(
                    title_text='<b>Proportion Solved</b>', title_font=dict(size=13, family=FONT_FAMILY, color="#0F172A"),
                    tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"), range=[-0.02, 1.05], showgrid=True, gridcolor='#F1F5F9', row=r_idx, col=c_idx
                )

            # Panel 6: Overall Aggregate Runtime ECDF (Mean across all problems)
            r_idx6, c_idx6 = coords[len(PROBLEM_IDS)]
            for s_name in solvers_to_plot:
                all_problem_ecdfs = []
                for p_id in PROBLEM_IDS:
                    cond = EvaluationCondition(dim=dim, noise_std=n_std, problem_id=p_id)
                    runs = all_benchmark_data.get(cond, {}).get(s_name, [])
                    if runs:
                        _, _, _, ecdf_curve = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                        all_problem_ecdfs.append(ecdf_curve)
                if all_problem_ecdfs:
                    mean_ecdf = np.mean(all_problem_ecdfs, axis=0)
                    s_style = get_solver_line_style(s_name)
                    fig_ecdf.add_trace(go.Scatter(
                        x=eval_grid, y=mean_ecdf, mode='lines', name=s_name,
                        line=dict(color=s_style['color'], width=s_style['width'], dash=s_style['dash']),
                        showlegend=False
                    ), row=r_idx6, col=c_idx6)
            fig_ecdf.update_xaxes(
                type='log', title_text='<b>Evaluations</b>', title_font=dict(size=13, family=FONT_FAMILY, color="#0F172A"),
                tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"), range=[0, 6.0],
                tickvals=tickvals, ticktext=ticktext,
                showgrid=True, gridcolor='#F1F5F9', row=r_idx6, col=c_idx6
            )
            fig_ecdf.update_yaxes(
                title_text='<b>Overall Proportion</b>', title_font=dict(size=13, family=FONT_FAMILY, color="#0F172A"),
                tickfont=dict(size=12, family=FONT_FAMILY, color="#1E293B"), range=[-0.02, 1.05], showgrid=True, gridcolor='#F1F5F9', row=r_idx6, col=c_idx6
            )

            for anno in fig_ecdf.layout.annotations:
                anno.update(font=dict(size=15, color='#0F172A', family=FONT_FAMILY))

            t_desc = f"{len(targets)} Adaptive Targets in {targets[0]:.1e} ≤ Δy ≤ {targets[-1]:.1e}"
            fig_ecdf.update_layout(
                template='plotly_white',
                title=dict(
                    text=f'<b>Empirical Runtime Cumulative Distribution Functions (ECDF) [{label_env}] — {model_name} ({dim}D)</b><br><span style="font-size:13px;color:#475569;font-weight:normal;">Proportion of Targets Solved ({t_desc}) vs. Function Evaluation Budget Across 5 BBOB Problem Classes (Budget = 1,000,000 evals)</span>',
                    font=dict(size=17, color='#0F172A', family=FONT_FAMILY),
                    x=0.02, y=0.97
                ),
                width=1340, height=860,
                margin=dict(l=70, r=40, t=110, b=120),
                legend=dict(
                    orientation='h',
                    yanchor='top', y=-0.14,
                    xanchor='center', x=0.5,
                    bgcolor='rgba(255,255,255,0.95)',
                    bordercolor='#E2E8F0',
                    borderwidth=1,
                    font=dict(size=13, family=FONT_FAMILY)
                )
            )
            out_m_ecdf = env_dir / 'target_precision_ecdf.png'
            fig_ecdf.write_image(str(out_m_ecdf), scale=3)

print('✅ 6-Panel Convergence and BBOB Runtime ECDF profiles generated successfully.')


### 3.2 Single-Algorithm Cross-Environment Direct Noise Overlays
Direct overlay curves comparing an individual algorithm's behavior in clean vs. noisy conditions to isolate noise degradation and search path deviation, saved in `results/cross_evaluation/`.

In [17]:
# ── THESIS: Single-Algorithm Cross-Environment Direct Noise Overlays ─────────
eval_grid = np.logspace(0, 6, 300)
tickvals = [1, 10, 100, 1000, 10000, 100000, 1000000]
ticktext = ['1', '10', '100', '1k', '10k', '100k', '1M']

# Primary algorithms to analyze individually
primary_solvers = [s for s in ALL_SOLVERS_ORDER if '(noise-adapted)' not in s]

coords = [((i // 3) + 1, (i % 3) + 1) for i in range(len(PROBLEM_IDS) + 1)]
subplot_titles = [f'<b>{BBOBFunction.get_name(p)}</b><br><sup>{BBOBFunction.get_class(p)}</sup>' for p in PROBLEM_IDS]
subplot_titles.append('<b>Overall Aggregate Profile</b><br><sup>Mean across 5 BBOB Problem Classes</sup>')

for dim in all_dims:
    for solver in primary_solvers:
        # Check if solver has any runs in this dimension
        solver_has_data = any(
            len(all_benchmark_data.get(EvaluationCondition(dim=dim, noise_std=n_std, problem_id=p_id), {}).get(solver, [])) > 0
            for p_id in PROBLEM_IDS
            for n_std in all_noise_stds
        )
        if not solver_has_data:
            continue

        s_slug = solver.lower().replace(' ', '_').replace('/', '_').replace('-', '_')
        envs = []
        for n_std in all_noise_stds:
            if n_std == clean_std:
                envs.append({
                    'name': f'Deterministic (σ={clean_std})',
                    'solver': solver,
                    'noise_std': clean_std,
                    'color': '#38BDF8',
                    'dash': 'solid'
                })
            else:
                envs.append({
                    'name': f'Zero-Shot Noisy (σ={n_std})',
                    'solver': solver,
                    'noise_std': n_std,
                    'color': get_noise_color(n_std, all_noise_stds),
                    'dash': 'solid'
                })
                # Check if noise-adapted exists in data
                adapted_solver = f'{solver} (noise-adapted)'
                cond_check = EvaluationCondition(dim=dim, noise_std=n_std, problem_id=PROBLEM_IDS[0])
                if adapted_solver in all_benchmark_data.get(cond_check, {}):
                    envs.append({
                        'name': f'Noise-Adapted (σ={n_std})',
                        'solver': adapted_solver,
                        'noise_std': n_std,
                        'color': '#10B981',
                        'dash': 'dash'
                    })
        
        target_dir = CROSS_EVAL_DIR / f'{dim}D' / s_slug
        target_dir.mkdir(parents=True, exist_ok=True)
        
        # 1. Multi-panel Convergence Direct Overlay
        fig_conv = make_subplots(
            rows=2, cols=3, subplot_titles=subplot_titles,
            vertical_spacing=0.18, horizontal_spacing=0.08
        )
        all_agg_vals = []
        for idx, p_id in enumerate(PROBLEM_IDS):
            r_idx, c_idx = coords[idx]
            for env in envs:
                cond = EvaluationCondition(dim=dim, noise_std=env['noise_std'], problem_id=p_id)
                runs = all_benchmark_data.get(cond, {}).get(env['solver'], [])
                if runs:
                    targets = service.compute_adaptive_targets(all_benchmark_data, noise_std=env['noise_std'])
                    mean_t, q25_t, q75_t, _ = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                    valid_pts = mean_t[np.isfinite(mean_t) & (mean_t > 0)]
                    if len(valid_pts) > 0:
                        all_agg_vals.extend(valid_pts.tolist())
                    
                    fig_conv.add_trace(go.Scatter(
                        x=eval_grid, y=mean_t, mode='lines', name=env['name'],
                        line=dict(color=env['color'], width=2.5, dash=env['dash']),
                        showlegend=(idx == 0)
                    ), row=r_idx, col=c_idx)
                    
                    hex_c = env['color'].lstrip('#')
                    rgb_fill = f'rgba({int(hex_c[0:2],16)}, {int(hex_c[2:4],16)}, {int(hex_c[4:6],16)}, 0.15)'
                    fig_conv.add_trace(go.Scatter(
                        x=np.concatenate([eval_grid, eval_grid[::-1]]),
                        y=np.concatenate([q75_t, q25_t[::-1]]),
                        fill='toself', fillcolor=rgb_fill,
                        line=dict(color='rgba(255,255,255,0)'),
                        showlegend=False, hoverinfo='skip'
                    ), row=r_idx, col=c_idx)
        
        # Aggregate Panel 6
        r_idx6, c_idx6 = coords[len(PROBLEM_IDS)]
        for env in envs:
            all_means = []
            for p_id in PROBLEM_IDS:
                cond = EvaluationCondition(dim=dim, noise_std=env['noise_std'], problem_id=p_id)
                runs = all_benchmark_data.get(cond, {}).get(env['solver'], [])
                if runs:
                    targets = service.compute_adaptive_targets(all_benchmark_data, noise_std=env['noise_std'])
                    mean_t, _, _, _ = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                    all_means.append(mean_t)
            if all_means:
                agg_mean = np.mean(all_means, axis=0)
                fig_conv.add_trace(go.Scatter(
                    x=eval_grid, y=agg_mean, mode='lines', name=env['name'],
                    line=dict(color=env['color'], width=3.0, dash=env['dash']),
                    showlegend=False
                ), row=r_idx6, col=c_idx6)
        
        if all_agg_vals:
            min_val = max(np.percentile(all_agg_vals, 1), 1e-16)
            max_val = max(np.percentile(all_agg_vals, 99) * 2.0, 10.0)
            y_range = [np.floor(np.log10(min_val)), np.ceil(np.log10(max_val))]
        else:
            y_range = [-16, 6]
            
        fig_conv.update_xaxes(type='log', range=[0, 6.0], tickvals=tickvals, ticktext=ticktext, title_text='<b>Function Evaluations (Budget = 10⁶)</b>', showgrid=True, gridcolor='#F1F5F9')
        fig_conv.update_yaxes(type='log', range=y_range, title_text='<b>Error Δy (Log Scale)</b>', showgrid=True, gridcolor='#F1F5F9')
        fig_conv.update_layout(
            template='plotly_white',
            title=dict(
                text=f'<b>Cross-Environment Noise Direct Overlay: {solver} ({dim}D)</b><br><span style="font-size:13px;color:#475569;">Evaluation Trajectory Degradation and Search Dynamics Across Noise Environments</span>',
                font=dict(size=18, family=FONT_FAMILY, color='#0F172A'), x=0.02, y=0.98
            ),
            width=1480, height=920, margin=dict(l=70, r=40, t=110, b=80),
            legend=dict(orientation='h', yanchor='top', y=-0.08, xanchor='center', x=0.5, bgcolor='rgba(255,255,255,0.95)', bordercolor='#E2E8F0', borderwidth=1, font=dict(size=13, family=FONT_FAMILY))
        )
        fig_conv.write_image(str(target_dir / 'convergence_noise_overlay.png'), scale=2)
        
        # 2. Multi-panel ECDF Direct Overlay
        fig_ecdf = make_subplots(
            rows=2, cols=3, subplot_titles=subplot_titles,
            vertical_spacing=0.18, horizontal_spacing=0.08
        )
        for idx, p_id in enumerate(PROBLEM_IDS):
            r_idx, c_idx = coords[idx]
            for env in envs:
                cond = EvaluationCondition(dim=dim, noise_std=env['noise_std'], problem_id=p_id)
                runs = all_benchmark_data.get(cond, {}).get(env['solver'], [])
                if runs:
                    targets = service.compute_adaptive_targets(all_benchmark_data, noise_std=env['noise_std'])
                    _, _, _, ecdf_t = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                    fig_ecdf.add_trace(go.Scatter(
                        x=eval_grid, y=ecdf_t, mode='lines', name=env['name'],
                        line=dict(color=env['color'], width=2.5, dash=env['dash']),
                        showlegend=(idx == 0)
                    ), row=r_idx, col=c_idx)
        
        for env in envs:
            all_ecdfs = []
            for p_id in PROBLEM_IDS:
                cond = EvaluationCondition(dim=dim, noise_std=env['noise_std'], problem_id=p_id)
                runs = all_benchmark_data.get(cond, {}).get(env['solver'], [])
                if runs:
                    targets = service.compute_adaptive_targets(all_benchmark_data, noise_std=env['noise_std'])
                    _, _, _, ecdf_t = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                    all_ecdfs.append(ecdf_t)
            if all_ecdfs:
                agg_ecdf = np.mean(all_ecdfs, axis=0)
                fig_ecdf.add_trace(go.Scatter(
                    x=eval_grid, y=agg_ecdf, mode='lines', name=env['name'],
                    line=dict(color=env['color'], width=3.0, dash=env['dash']),
                    showlegend=False
                ), row=r_idx6, col=c_idx6)
                
        fig_ecdf.update_xaxes(type='log', range=[0, 6.0], tickvals=tickvals, ticktext=ticktext, title_text='<b>Function Evaluations (Budget = 10⁶)</b>', showgrid=True, gridcolor='#F1F5F9')
        fig_ecdf.update_yaxes(range=[0, 1.05], title_text='<b>Proportion Solved</b>', showgrid=True, gridcolor='#F1F5F9')
        fig_ecdf.update_layout(
            template='plotly_white',
            title=dict(
                text=f'<b>Cross-Environment Target ECDF Direct Overlay: {solver} ({dim}D)</b><br><span style="font-size:13px;color:#475569;">Empirical Runtime ECDF Degradation Across Noise Environments</span>',
                font=dict(size=18, family=FONT_FAMILY, color='#0F172A'), x=0.02, y=0.98
            ),
            width=1480, height=920, margin=dict(l=70, r=40, t=110, b=80),
            legend=dict(orientation='h', yanchor='top', y=-0.08, xanchor='center', x=0.5, bgcolor='rgba(255,255,255,0.95)', bordercolor='#E2E8F0', borderwidth=1, font=dict(size=13, family=FONT_FAMILY))
        )
        fig_ecdf.write_image(str(target_dir / 'ecdf_noise_overlay.png'), scale=2)

print('✅ Completed Cross-Environment Single-Algorithm Direct Noise Overlays.')


2026-09-13 10:56:28 INFO Chromium init'ed with kwargs {}
2026-09-13 10:56:28 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-13 10:56:28 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp4zojye35.
2026-09-13 10:56:28 INFO Opening browser.
2026-09-13 10:56:28 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpvysd8u98.
2026-09-13 10:56:28 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpvysd8u98
2026-09-13 10:56:29 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp4zojye35/index.html
2026-09-13 10:56:29 INFO Getting tab from queue (has 1)
2026-09-13 10:56:29 INFO Got 10A1
2026-09-13 10:56:30 INFO Reloading tab 10A1 before return.
2026-09-13 10:56:30 INFO Putting tab 10A1 back (queue size: 0).
2026-09-13 10:56:30 INFO Waiting for all cleanups to finish.
2026-09-13 10:56:30 INFO Exiting Kaleido.
2026-09-13 10:56:30 INFO T

✅ Completed Cross-Environment Single-Algorithm Direct Noise Overlays.


---
## Section 4: Algorithmic Failure Analysis & Stagnation Matrices

### 4.1 Figure 10: Multi-Tier Algorithmic Outcome Breakdown & Dimensional Diagnostics
Quantifies algorithmic convergence outcomes into four empirical precision tiers:
1. **High Precision Success** ($\Delta y \le 10^{-8}$)
2. **Medium Precision** ($10^{-8} < \Delta y \le 10^{-2}$)
3. **Low Precision / Stagnation** ($10^{-2} < \Delta y \le 10^{2}$)
4. **Severe Divergence** ($\Delta y > 10^{2}$ or Timeout)

Exports `fig_10a_algorithmic_failure_breakdown.png` and `fig_10c_algorithmic_failure_by_dimension.png` into `results/failure_analysis/`.

In [18]:
# ── THESIS Figure 10: Multi-Tier Failure Breakdown & Partitioned Failure Matrices ──
tiers_df = service.compute_convergence_tiers(all_benchmark_data)
tiers_df["Canonical Solver"] = tiers_df["Solver"].str.replace(r" \(noise-adapted\)", "", regex=True)

TIER_ORDER = [
    'High Precision (Δy ≤ 10⁻⁸)',
    'Moderate Convergence (10⁻⁸ < Δy ≤ 10⁻²)',
    'Minor Progress (10⁻² < Δy ≤ 1.0)',
    'Severe Stagnation / Failure (Δy > 1.0)'
]

TIER_COLORS = {
    'High Precision (Δy ≤ 10⁻⁸)': '#38BDF8',           # Soft Sky Blue
    'Moderate Convergence (10⁻⁸ < Δy ≤ 10⁻²)': '#34D399', # Soft Mint
    'Minor Progress (10⁻² < Δy ≤ 1.0)': '#FBBF24',         # Soft Amber Gold
    'Severe Stagnation / Failure (Δy > 1.0)': '#F87171'    # Soft Coral
}

soft_colorscale = [
    [0.0, '#F8FAFC'],
    [0.2, '#FEF3C7'],
    [0.5, '#FDE68A'],
    [0.75, '#FCA5A5'],
    [1.0, '#F87171']
]

# ── 1. Figure 10A: Overall Outcome Breakdown ───────────────────────────────
solvers_order = (
    tiers_df[tiers_df['Tier'] == 'High Precision (Δy ≤ 10⁻⁸)']
    .groupby('Canonical Solver').size()
    / tiers_df.groupby('Canonical Solver').size()
).fillna(0.0).sort_values(ascending=True).index.tolist()

fig10a = go.Figure()
for tier in TIER_ORDER:
    tier_counts = tiers_df[tiers_df['Tier'] == tier].groupby('Canonical Solver').size()
    total_counts = tiers_df.groupby('Canonical Solver').size()
    tier_pcts = [(tier_counts.get(s, 0) / total_counts.get(s, 1)) * 100.0 for s in solvers_order]
    fig10a.add_trace(go.Bar(
        y=solvers_order,
        x=tier_pcts,
        name=tier,
        orientation='h',
        marker=dict(color=TIER_COLORS[tier], line=dict(color='rgba(15, 23, 42, 0.3)', width=0.8)),
        text=[f'{p:.1f}%' if p >= 5.0 else '' for p in tier_pcts],
        textposition='inside',
        insidetextanchor='middle',
        textfont=dict(size=11, family=FONT_FAMILY, color='#0F172A')
    ))

h_10a = max(720, len(solvers_order) * 28 + 180)
fig10a.update_layout(
    template='plotly_white',
    barmode='stack',
    title=dict(
        text='<b>Figure 10A: Multi-Tier Algorithmic Outcome & Failure Breakdown Across All Conditions</b><br><span style="font-size:13px;color:#475569;font-weight:normal;">Distribution of Convergence Tiers (Target Hits vs. Stagnation) Across All Evaluated Black-Box Optimizers</span>',
        font=dict(size=18, color='#0F172A', family=FONT_FAMILY),
        x=0.02, y=0.97
    ),
    width=1380, height=h_10a,
    margin=dict(l=220, r=40, t=110, b=90),
    legend=dict(
        orientation='h', yanchor='top', y=-0.10, xanchor='center', x=0.5,
        bgcolor='rgba(255,255,255,0.95)', bordercolor='#E2E8F0', borderwidth=1,
        font=dict(size=12, family=FONT_FAMILY)
    ),
    xaxis=dict(
        title='<b>Proportion of Evaluation Runs (%)</b>',
        title_font=dict(size=14, family=FONT_FAMILY, color='#0F172A'),
        tickfont=dict(size=12, family=FONT_FAMILY, color='#1E293B'),
        range=[0, 100], showgrid=True, gridcolor='#F1F5F9'
    ),
    yaxis=dict(tickfont=dict(size=12, family=FONT_FAMILY, color='#1E293B'))
)

out_10a = FAILURE_DIR / "fig_10a_algorithmic_failure_breakdown.png"
fig10a.write_image(str(out_10a), scale=3)
print('✅ Figure 10A (Overall Failure Breakdown) generated in results/failure_analysis/')

# ── 2. Figure 10B: Failure Matrix Across BBOB Classes ───────────────────────
prob_ids = [p for p in [1, 8, 11, 15, 21] if p in PROBLEM_IDS]
prob_labels = [f'{BBOBFunction.get_name(p)}<br>(f{p})' for p in prob_ids]

solvers_order_desc = solvers_order[::-1]
matrix_all = np.zeros((len(solvers_order_desc), len(prob_ids)))
for r, s in enumerate(solvers_order_desc):
    for c, p in enumerate(prob_ids):
        cell_data = tiers_df[(tiers_df['Canonical Solver'] == s) & (tiers_df['Problem ID'] == p)]
        if not cell_data.empty:
            fail_count = (cell_data['Best Error'] > 1.0).sum()
            matrix_all[r, c] = (fail_count / len(cell_data)) * 100.0
        else:
            matrix_all[r, c] = 0.0

fig10b = go.Figure(data=go.Heatmap(
    z=matrix_all,
    x=prob_labels,
    y=solvers_order_desc,
    colorscale=soft_colorscale,
    zmin=0, zmax=100,
    text=[[f'{v:.0f}%' for v in row] for row in matrix_all],
    texttemplate='%{text}',
    textfont=dict(size=11, family=FONT_FAMILY, color='#0F172A'),
    colorbar=dict(
        title='<b>Failure Rate (%)</b>',
        title_font=dict(size=13, family=FONT_FAMILY),
        tickfont=dict(size=12, family=FONT_FAMILY),
        len=0.85
    )
))

h_10b = max(760, len(solvers_order_desc) * 28 + 180)
fig10b.update_layout(
    template='plotly_white',
    title=dict(
        text='<b>Figure 10B: Empirical Algorithmic Failure Rate Matrix Across BBOB Topologies</b><br><span style="font-size:13px;color:#475569;font-weight:normal;">Severe Stagnation / Failure Rate (Δy > 1.0) Across All Evaluated Black-Box Optimizers and BBOB Problem Classes</span>',
        font=dict(size=18, color='#0F172A', family=FONT_FAMILY),
        x=0.02, y=0.97
    ),
    width=1180, height=h_10b,
    margin=dict(l=220, r=40, t=110, b=90),
    xaxis=dict(tickfont=dict(size=11, family=FONT_FAMILY, color='#1E293B')),
    yaxis=dict(tickfont=dict(size=11, family=FONT_FAMILY, color='#1E293B'), autorange='reversed')
)

out_10b = FAILURE_DIR / "fig_10b_algorithmic_failure_matrix_heatmap.png"
fig10b.write_image(str(out_10b), scale=3)
print('✅ Figure 10B (Overall Failure Matrix Heatmap) generated in results/failure_analysis/')

# ── 3. Figure 10C: Breakdown Partitioned by Dimension (2D, 3D, 5D, 10D) ─────
dims = [d for d in [2, 3, 5, 10] if d in all_dims]
n_cols = 2 if len(dims) > 1 else 1
n_rows = (len(dims) + 1) // 2
fig10c = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f'<b>Dimension D = {d}</b>' for d in dims],
    horizontal_spacing=0.14, vertical_spacing=0.12,
    shared_yaxes=False
)
coords = [((i // n_cols) + 1, (i % n_cols) + 1) for i in range(len(dims))]

for idx, d in enumerate(dims):
    r_idx, c_idx = coords[idx]
    sub_d = tiers_df[tiers_df['Dim'] == d]
    d_solvers = (
        sub_d[sub_d['Tier'] == 'High Precision (Δy ≤ 10⁻⁸)']
        .groupby('Canonical Solver').size()
        / sub_d.groupby('Canonical Solver').size()
    ).fillna(0.0).sort_values(ascending=True).index.tolist()

    for tier_idx, tier in enumerate(TIER_ORDER):
        tier_counts = sub_d[sub_d['Tier'] == tier].groupby('Canonical Solver').size()
        total_counts = sub_d.groupby('Canonical Solver').size()
        tier_pcts = [(tier_counts.get(s, 0) / total_counts.get(s, 1)) * 100.0 for s in d_solvers]
        fig10c.add_trace(go.Bar(
            y=d_solvers,
            x=tier_pcts,
            name=tier,
            orientation='h',
            marker=dict(color=TIER_COLORS[tier], line=dict(color='rgba(15, 23, 42, 0.3)', width=0.8)),
            showlegend=(idx == 0),
            text=[f'{p:.0f}%' if p >= 8.0 else '' for p in tier_pcts],
            textposition='inside',
            insidetextanchor='middle',
            textfont=dict(size=10, family=FONT_FAMILY, color='#0F172A')
        ), row=r_idx, col=c_idx)

    fig10c.update_xaxes(range=[0, 100], title_text='<b>Run Percentage (%)</b>' if r_idx==n_rows else '', row=r_idx, col=c_idx)
    fig10c.update_yaxes(tickfont=dict(size=11, family=FONT_FAMILY, color='#1E293B'), row=r_idx, col=c_idx)

for anno in fig10c.layout.annotations:
    anno.update(font=dict(size=15, color='#0F172A', family=FONT_FAMILY))

fig10c.update_layout(
    template='plotly_white',
    barmode='stack',
    title=dict(
        text='<b>Figure 10C: Empirical Algorithmic Failure Breakdown Partitioned by Problem Dimension</b><br><span style="font-size:13px;color:#475569;font-weight:normal;">Distribution of Convergence Tiers and Stagnation Rates Across Search Dimensions</span>',
        font=dict(size=18, color='#0F172A', family=FONT_FAMILY),
        x=0.02, y=0.98
    ),
    width=1480, height=920,
    margin=dict(l=220, r=40, t=110, b=80),
    legend=dict(
        orientation='h', yanchor='top', y=-0.08, xanchor='center', x=0.5,
        bgcolor='rgba(255,255,255,0.95)', bordercolor='#E2E8F0', borderwidth=1,
        font=dict(size=12, family=FONT_FAMILY)
    )
)

out_10c = FAILURE_DIR / "fig_10c_algorithmic_failure_by_dimension.png"
fig10c.write_image(str(out_10c), scale=3)
print('✅ Figure 10C (Failure by Dimension) generated in results/failure_analysis/')


2026-09-13 11:02:36 INFO TemporaryDirectory.cleanup() worked.
2026-09-13 11:02:36 INFO shutil.rmtree worked.
2026-09-13 11:02:36 INFO TemporaryDirectory.cleanup() worked.
2026-09-13 11:02:36 INFO shutil.rmtree worked.
2026-09-13 11:02:37 INFO Chromium init'ed with kwargs {}
2026-09-13 11:02:37 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-13 11:02:37 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpgl8w6q1p.
2026-09-13 11:02:37 INFO Opening browser.
2026-09-13 11:02:37 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpvmc98hhc.
2026-09-13 11:02:37 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpvmc98hhc
2026-09-13 11:02:37 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpgl8w6q1p/index.html
2026-09-13 11:02:38 INFO Getting tab from queue (has 1)
2026-09-13 11:02:38 INFO Got 8BC5
2026-09-13 11:02:38 INFO Reloading

✅ Figure 10A (Overall Failure Breakdown) generated in results/failure_analysis/
✅ Figure 10B (Overall Failure Matrix Heatmap) generated in results/failure_analysis/


2026-09-13 11:02:40 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpf9ctyrev/index.html
2026-09-13 11:02:41 INFO Getting tab from queue (has 1)
2026-09-13 11:02:41 INFO Got EC0A
2026-09-13 11:02:41 INFO Reloading tab EC0A before return.
2026-09-13 11:02:41 INFO Putting tab EC0A back (queue size: 0).
2026-09-13 11:02:41 INFO Waiting for all cleanups to finish.
2026-09-13 11:02:41 INFO Exiting Kaleido.
2026-09-13 11:02:41 INFO TemporaryDirectory.cleanup() worked.
2026-09-13 11:02:41 INFO shutil.rmtree worked.
2026-09-13 11:02:41 INFO Closing browser.
2026-09-13 11:02:41 INFO TemporaryDirectory.cleanup() worked.
2026-09-13 11:02:41 INFO shutil.rmtree worked.
2026-09-13 11:02:41 INFO Closing browser.
2026-09-13 11:02:41 INFO Cancelling tasks.
2026-09-13 11:02:41 INFO Exiting Kaleido/Choreo.
2026-09-13 11:02:41 INFO TemporaryDirectory.cleanup() worked.
2026-09-13 11:02:41 INFO shutil.rmtree worked.
2026-09-13 11:02:41 INFO Cancelling tasks.
2026-09-13 11:02:4

✅ Figure 10C (Failure by Dimension) generated in results/failure_analysis/
